## Session 8, in class: profiling, and one equilibrium wrapper

Computational Macro (HWS 2026), University of Mannheim. The coding session after the midterm. Not graded.

Last week's notebook produced the same number two ways, by Monte Carlo simulation and by Young's method.
Part 1 of this notebook is that code again, given, run through once so that everything is defined. Part 2
measures: the two methods against each other, the three pieces of $A(r)$, and the profiler of the standard
library (the session 8 slides). Part 3 wraps the distribution into one general equilibrium solver that handles
Huggett and Aiyagari alike. Part 4 is yours: change the settings of the solver and watch what it costs.

**How to work with this notebook**

- Run the cells from top to bottom (`Shift + Enter`). After a restart of the kernel, start again at the top.
- There are **no gaps** this week. The code is complete; the work is to run it, read the output, and
  understand it: every timing of the slides is reproduced here on your machine.
- Part 1 has nothing to read: it is last week's code. Run it and move on.
- Each part ends with a **Check**: a number to compare with, from the slides.
- Part 4 has no check. It asks you to change the settings and read the table you get.

**Plan**

| Part | Content | Slides | Time |
| --- | --- | --- | --- |
| 1 | Last week's code: the household, the simulation, Young's method (given) | S6 | 5 min |
| 2 | Profiling: Monte Carlo against Young, the three pieces of $A(r)$, what takes so long | S8: 2 to 8 | 20 min |
| 3 | One equilibrium wrapper for Huggett and Aiyagari | S6: 18 to 22 | 15 min |
| 4 | Experiment: the settings of the solver, and what they cost | S8: 2, 3 | 20 min |


**Toolbox: the few tools this notebook uses that are new**

| Tool | What it does | Example |
| --- | --- | --- |
| `np.searchsorted(grid, x, side="right") - 1` | the index `l` of the grid point just below `x` (for a sorted `grid`) | locating the lottery |
| `np.clip(x, lo, hi)` | elementwise: `x`, but not below `lo` and not above `hi` | keep `l` inside `0, ..., N_a - 2` |
| `np.arange(n).reshape(nz, na).T` | the numbers `0, ..., n - 1` as an `na` by `nz` table, counting down the columns | `idx[i, m]` is the row of state $(a_i, z_m)$ in `Q` |
| `sp.csr_matrix((vals, (rows, cols)), shape=(n, n))` | a sparse matrix from triplets: `vals[k]` sits at `(rows[k], cols[k])` | $\mathbf{Q}_{\mathbf{g}}$ with $2 N_z$ entries per row |
| `Q.T @ lam` | sparse matrix times vector | one step of the distribution |
| `rng = np.random.default_rng(seed)`, `rng.random(N)` | a seeded random number generator, `N` uniforms | the simulation |
| `time.perf_counter()` | a clock for timing a piece of code, in seconds | part 2 |
| `cProfile.Profile()`, `pstats.Stats` | the profiler of the standard library: time per function | part 2 |

Everything else is from the earlier notebooks: `A @ B`, `x[:, None]`, `np.interp`, `A.sum(axis=1)`.
A policy is stored as an $N_a \times N_z$ array: row `i` is the asset grid point $a_i$, column `m` the income state $z_m$.

### 0. Packages

Nothing new to install: `scipy` (for the sparse matrix) and `quantecon` (for the income chain) are in the
course environment since weeks 3 and 5, `time`, `cProfile`, and `pstats` ship with Python. The first cell
installs the packages into the environment of the running kernel if they are missing.

In [ ]:
# Installs the packages of this notebook into the environment of the running kernel.
%pip install "numpy==2.2.*" "scipy==1.15.*" "matplotlib==3.10.*" "quantecon>=0.7"

In [ ]:
import time
import cProfile
import pstats
import warnings
from dataclasses import dataclass
from typing import Callable

import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import quantecon as qe

warnings.filterwarnings("ignore", message="The API of rouwenhorst")   # the notice on every rouwenhorst call, as in week 5

### 1. Last week's code, given

The household of the session 6 slides, the Monte Carlo simulation, and Young's method, exactly as in
`Week_7/aggregation_and_young`, with the gaps filled. Nothing to read here that you have not seen, run the
cells and check that the last one prints mean assets $8.517$.

In [ ]:
@dataclass(frozen=True)
class EconomicParameters:
    gamma: float = 2.0        # relative risk aversion
    beta: float = 0.96        # discount factor
    rho: float = 0.9          # persistence of ln z
    sig_eps: float = 0.2      # standard deviation of the innovation to ln z


@dataclass(frozen=True)
class NumericalParameters:
    na: int = 200             # asset grid points N_a
    nz: int = 7               # income grid points N_z
    amax: float = 100.0       # upper grid bound a_{N_a}
    crit_pol: float = 1e-8    # the EGM stops when the policy moves by less
    crit_dist: float = 1e-12  # the distribution iteration stops when lam moves by less (part 3)
    maxiter: int = 100_000    # iteration cap, the safety net

In [ ]:
@dataclass(frozen=True)
class Grids:
    a: np.ndarray             # asset grid, N_a
    z: np.ndarray             # income grid, N_z
    Pi: np.ndarray            # transition matrix, N_z x N_z
    lam_z: np.ndarray         # stationary distribution of the income chain, N_z


def make_grids(amin, par, mpar):
    # The income chain of session 4 on N_z Rouwenhorst states (week 5, part 1), income levels z = exp(ln z),
    # and the quadratically spaced asset grid: a_i = amin + (amax - amin) s_i^2 with s_i equidistant on [0, 1]
    mc = qe.rouwenhorst(mpar.nz, par.rho, par.sig_eps)   # the chain for ln z, as in week 5
    z = np.exp(mc.state_values)                          # income levels
    lam_z = mc.stationary_distributions[0]               # the stationary distribution of the chain, week 4
    s = np.linspace(0.0, 1.0, mpar.na)                   # N_a equidistant points on [0, 1]
    a = amin + (mpar.amax - amin) * s ** 2               # squared: dense near amin, sparse near amax
    return Grids(a=a, z=z, Pi=mc.P, lam_z=lam_z)

In [ ]:
par, mpar = EconomicParameters(), NumericalParameters()  # the default values listed above
gri = make_grids(0.0, par, mpar)                         # borrowing limit a >= 0 for parts 1 and 2
print("income states:", np.round(gri.z, 3))
print("stationary distribution of income:", np.round(gri.lam_z, 3), " mean income E[z] =", round(gri.lam_z @ gri.z, 4))
print("asset grid: first spacing", round(gri.a[1] - gri.a[0], 4), " last spacing", round(gri.a[-1] - gri.a[-2], 3))

In [ ]:
def uprime(c, par):
    # marginal utility of CRRA utility u(c) = c^(1 - gamma) / (1 - gamma), that is u'(c) = c^(-gamma)
    return c ** (-par.gamma)                    # works on numbers and arrays


def uprime_inv(m, par):
    # inverse of marginal utility: the c with u'(c) = m
    return m ** (-1 / par.gamma)

In [ ]:
def egm_step(C, r, w, par, gri):
    # One step of the endogenous grid method (session 5, slide 20): from the policy C to the new policy on the grid
    R = 1 + r
    EMU = uprime(C, par) @ gri.Pi.T             # expected marginal utility tomorrow at every (a'_l, z_m), week 5 gap 5
    ctilde = uprime_inv(par.beta * R * EMU, par)  # (a) consumption today from the inverted Euler equation
    atilde = (ctilde + gri.a[:, None] - w * gri.z[None, :]) / R   # (b) today's assets from the budget constraint
    Cnew = np.empty_like(C)                     # (c) back to the grid, one income state at a time
    for m in range(len(gri.z)):
        Cnew[:, m] = np.interp(gri.a, atilde[:, m], ctilde[:, m])
        below = gri.a < atilde[0, m]            # poorer than the household that saves a' = a_1: the limit binds
        Cnew[below, m] = w * gri.z[m] + R * gri.a[below] - gri.a[0]   # consume everything down to a' = a_1
    return Cnew

In [ ]:
def solve_household(r, w, par, mpar, gri, C0=None):
    # The EGM loop of week 5: iterate egm_step until the policy stops moving. Returns C, the savings policy G, and the count.
    # C0: an optional starting guess for the policy (the solution at nearby prices, part 2)
    R = 1 + r
    C = np.maximum(r * gri.a[:, None] + w * gri.z[None, :], 1e-6) if C0 is None else C0.copy()   # the guess: consume interest and income
    for it in range(1, mpar.maxiter + 1):
        Cnew = egm_step(C, r, w, par, gri)
        dist = np.max(np.abs(Cnew - C))         # largest change of the policy
        C = Cnew
        if dist < mpar.crit_pol:
            break
    G = w * gri.z[None, :] + R * gri.a[:, None] - C   # savings a' = w z + R a - c, N_a x N_z
    return C, G, it

In [ ]:
r0, w0 = 0.03, 1.0                              # the prices of parts 1 and 2
t0 = time.perf_counter()
C0, G0, it0 = solve_household(r0, w0, par, mpar, gri)
print(f"EGM: {it0} iterations, {time.perf_counter() - t0:.3f} s")
print("net saving g(a, z) - a at a = 0, by income state:", np.round(G0[0] - gri.a[0], 3))
i10 = np.argmin(np.abs(gri.a - 10))             # the grid point closest to a = 10
print(f"net saving at a = {gri.a[i10]:.2f}, by income state:", np.round(G0[i10] - gri.a[i10], 3))

In [ ]:
def simulate_panel(G, gri, N, T, seed):
    # N households for T periods. Returns the final cross section (assets, income index) and the mean of assets in every period
    rng = np.random.default_rng(seed)           # seeded, so that the run is reproducible
    cum = np.cumsum(gri.Pi, axis=1)
    a = np.full(N, gri.a[0])                    # everybody starts at the limit
    m = rng.choice(len(gri.z), size=N, p=gri.lam_z)   # income indices drawn from the stationary distribution
    means = np.empty(T)                         # mean assets in every period
    for t in range(T):
        anew = np.empty(N)
        for k in range(len(gri.z)):             # (i) tomorrow's assets: one interpolation per income state
            sel = m == k                        # the households with income z_k today
            anew[sel] = np.interp(a[sel], gri.a, G[:, k])
        a = anew
        u = rng.random(N)                       # (ii) tomorrow's income: one uniform per household
        m = np.minimum((u[:, None] > cum[m]).sum(axis=1), len(gri.z) - 1)   # the first n with F_mn >= u, for every household at once (week 5 by hand)
        means[t] = a.mean()
    return a, m, means

In [ ]:
N, T = 10_000, 1_000
t0 = time.perf_counter()
a_sim, m_sim, means = simulate_panel(G0, gri, N, T, seed=11)
t_sim = time.perf_counter() - t0
print(f"simulation of N = {N} households for T = {T} periods: {t_sim:.2f} s")
print(f"mean assets {a_sim.mean():.3f}, share at the limit {(a_sim <= gri.a[0] + 1e-10).mean():.3f}, median {np.median(a_sim):.2f}")
print("mean assets over time, t = 10, 50, 100, 300, 1000:", np.round(means[[9, 49, 99, 299, 999]], 2))

In [ ]:
mc_rows = []                                    # (N, mean over seeds, sd across seeds, seconds per simulation), for part 2
for Nn in (1_000, 10_000):
    ms, tt = [], time.perf_counter()
    for seed in range(3):
        a_s, _, _ = simulate_panel(G0, gri, Nn, 500, seed=seed)
        ms.append(a_s.mean())
    tt = (time.perf_counter() - tt) / 3
    mc_rows.append((Nn, np.mean(ms), np.std(ms), tt))
    print(f"N = {Nn:7d}: mean assets over 3 seeds {np.mean(ms):.3f}, sd across seeds {np.std(ms):.3f}, {tt:.2f} s per simulation")

In [ ]:
def lottery(G, gri):
    # For every policy value a'_im: the index l of the grid point just below it, and the weight w_im on the point above
    na = len(gri.a)
    l = np.clip(np.searchsorted(gri.a, G, side="right") - 1, 0, na - 2)   # locate: a_l <= a'_im < a_{l+1}, l at most N_a - 2
    w = (G - gri.a[l]) / (gri.a[l + 1] - gri.a[l])
    w = np.clip(w, 0.0, 1.0)                    # exactly at the top of the grid, w would be 1 + 1e-16
    return l, w

In [ ]:
l0, w0_ = lottery(G0, gri)
print("weights inside [0, 1]:", (w0_ >= 0).all() and (w0_ <= 1).all())
# The lottery reproduces the policy, except where the richest households at a = 100 would save beyond the grid
# (two points; the lottery caps them at a_N, and part 3 checks that no mass ever gets there)
print("lottery mean equals the policy (capped at a_N):", np.allclose((1 - w0_) * gri.a[l0] + w0_ * gri.a[l0 + 1], np.minimum(G0, gri.a[-1])))
print("policy values above the top of the grid:", (G0 > gri.a[-1]).sum())
i17 = np.argmin(np.abs(gri.a - 17.0))           # the example of slide 12: a = 17.0, income z_6
print(f"a = {gri.a[i17]:.2f}, z_6: a' = {G0[i17, 5]:.2f} lies between {gri.a[l0[i17, 5]]:.2f} and {gri.a[l0[i17, 5] + 1]:.2f}, w = {w0_[i17, 5]:.2f}")

In [ ]:
# The row of Q for the household of the example, written out: destination, share, as on slide 13
i, m = i17, 5                                   # the cell (a_i, z_m) = (17.0, z_6)
l_im, w_im = l0[i, m], w0_[i, m]
rows = []
for n in range(len(gri.z)):                     # income tomorrow
    rows.append((gri.a[l_im], gri.z[n], gri.Pi[m, n] * (1 - w_im)))
    rows.append((gri.a[l_im + 1], gri.z[n], gri.Pi[m, n] * w_im))   # the share landing on (a_{l+1}, z_n)
print(f"mass leaving (a, z) = ({gri.a[i]:.2f}, z_6), policy a' = {G0[i, m]:.2f}, lottery between {gri.a[l_im]:.2f} and {gri.a[l_im + 1]:.2f} with w = {w_im:.2f}")
print("   a' tomorrow   z tomorrow   share")
for a_to, z_to, share in rows:
    if share > 1e-4:                            # the chain never jumps more than two income states in practice, skip the tiny ones
        print(f"   {a_to:9.2f}   {z_to:9.2f}   {share:.4f}")
print(f"shares add up to {sum(s for _, _, s in rows):.6f}, mean of a' tomorrow {sum(a_to * s for a_to, _, s in rows):.4f} (the policy says {G0[i, m]:.4f})")

In [ ]:
def transition_matrix(G, gri):
    # Q[(i, m), (l, n)] = Pi[m, n] (1 - w_im) and Q[(i, m), (l + 1, n)] = Pi[m, n] w_im: the policy chain as a sparse matrix
    na, nz = len(gri.a), len(gri.z)
    idx = np.arange(na * nz).reshape(nz, na).T  # idx[i, m] = i + m N_a: the row of state (a_i, z_m), the order of Howard's step (week 5)
    l, w = lottery(G, gri)                      # the lottery of every (i, m)
    rows, cols, vals = [], [], []               # the triplets, collected column block by column block
    for m in range(nz):                         # income today
        for n in range(nz):                     # income tomorrow
            # mass 1 - w_im goes to (a_l, z_n) ...
            rows.append(idx[:, m]); cols.append(idx[l[:, m], n]); vals.append(gri.Pi[m, n] * (1 - w[:, m]))
            # ... and mass w_im to (a_{l+1}, z_n)
            rows.append(idx[:, m]); cols.append(idx[l[:, m] + 1, n]); vals.append(gri.Pi[m, n] * w[:, m])
    return sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(na * nz, na * nz))

In [ ]:
t0 = time.perf_counter()
Q0 = transition_matrix(G0, gri)
print(f"Q built in {1000 * (time.perf_counter() - t0):.1f} ms: shape {Q0.shape}, {Q0.nnz} stored entries, "
      f"{100 * Q0.nnz / Q0.shape[0] ** 2:.2f} percent of the matrix")
rowsums = np.asarray(Q0.sum(axis=1)).ravel()
print(f"rows sum to one: largest deviation {np.max(np.abs(rowsums - 1)):.1e}")
idx0 = np.arange(mpar.na * mpar.nz).reshape(mpar.nz, mpar.na).T   # the same table, for looking up rows below
row = Q0[idx0[i17, 5]].toarray().ravel()        # the row of the example cell, as a dense vector
print(f"row of (17.0, z_6): {np.count_nonzero(row)} nonzero entries, largest {row.max():.4f}, sum {row.sum():.6f}")

In [ ]:
def stationary_dist(Q, mpar, gri, lam0=None):
    # Iterate lam_{n+1} = Q^T lam_n from the uniform distribution (or from lam0). Returns lam as an N_a x N_z table, and the count
    n = Q.shape[0]
    lam = np.full(n, 1 / n) if lam0 is None else lam0.T.ravel()   # the start, as a vector in the order of idx
    QT = Q.T.tocsr()                            # the transpose once, in the fast row format
    for it in range(1, mpar.maxiter + 1):
        lam_new = QT @ lam
        dist = np.max(np.abs(lam_new - lam))    # largest change
        lam = lam_new
        if dist < mpar.crit_dist:
            break
    return lam.reshape(len(gri.z), len(gri.a)).T, it   # back to the (a, z) table, the same order as idx

In [ ]:
t0 = time.perf_counter()
lam0, itl = stationary_dist(Q0, mpar, gri)
t_young = time.perf_counter() - t0
print(f"Young's iteration: {itl} iterations to 1e-12, {1000 * t_young:.0f} ms")
# One step by hand, for one cell: the mass arriving at the example cell (17.0, z_6) is the weighted column sum of slide 14
lam_vec = lam0.T.ravel()                        # lam as a vector in the order of idx
col = Q0[:, idx0[i17, 5]].toarray().ravel()     # column (17.0, z_6) of Q: who sends mass there, and how much
print(f"mass arriving at (17.0, z_6): {col @ lam_vec:.3e} from {np.count_nonzero(col)} cells, against lam there {lam0[i17, 5]:.3e} (equal: stationary)")

In [ ]:
def mean_assets(lam, gri):
    # A = sum_{i, m} lam_im a_i, with lam as an N_a x N_z table
    return (lam * gri.a[:, None]).sum()

In [ ]:
A0 = mean_assets(lam0, gri)
lam_a0 = lam0.sum(axis=1)                       # the marginal distribution of assets
print(f"mean assets {A0:.4f} (simulation: {a_sim.mean():.4f})")
print(f"share at the limit {lam0[0].sum():.4f} (simulation: {(a_sim <= gri.a[0] + 1e-10).mean():.4f})")
print(f"check 1, mass: {lam0.sum():.12f}")
print(f"check 2, marginal over assets against lam_z: {np.max(np.abs(lam0.sum(axis=0) - gri.lam_z)):.1e}")
print(f"check 3, mass at the top of the grid: {lam0[-1].sum():.1e}")
print("mean assets by income state:", np.round((lam0 * gri.a[:, None]).sum(axis=0) / lam0.sum(axis=0), 1))
cdf0 = np.cumsum(lam_a0)
print(f"median {gri.a[np.searchsorted(cdf0, 0.5)]:.1f}, share below one year of mean income: {cdf0[np.searchsorted(gri.a, gri.lam_z @ gri.z) - 1]:.2f}")

### 2. Profiling: what takes so long?

*Session 8, slides 2 to 8.*

Last week's notebook produced the same number two ways. Before choosing, **measure**: guessing where the time goes is
the most common mistake in numerical work. Three levels, from cheap to thorough.

1. **A clock around a call.** `time.perf_counter()` before and after (Julia: `@time`, which also reports
   allocations). Repeat the call a few times and take the best: the first call of anything is slow (caches,
   and in Julia the compilation), and the machine does other things in the background.
2. **Clocks around the pieces.** $A(r)$ has three pieces: solve the household, build $\mathbf{Q}_{\mathbf{g}}$,
   iterate on $\boldsymbol{\lambda}$. Time them one by one and the picture is usually clear.
3. **A profiler.** It samples (Julia) or records (Python) which function is running and reports the time
   per function, with the calls inside counted separately. Use it when the piecewise clocks are not enough,
   or when the code is somebody else's.

**Monte Carlo against Young, in one table.** The first comparison is the one that matters: how much does one
digit of the mean cost?

In [ ]:
def best_of(f, reps=3):
    # the result of f() and the best of `reps` timings in seconds (the first call is often slower: caches, and in Julia, compilation)
    best = np.inf
    for _ in range(reps):
        t0 = time.perf_counter()
        out = f()
        best = min(best, time.perf_counter() - t0)
    return out, best

In [ ]:
(lam_y, _), t_y = best_of(lambda: stationary_dist(transition_matrix(G0, gri), mpar, gri))
print(f"{'method':>24}  {'mean assets':>12}  {'sd across seeds':>16}  {'seconds':>8}")
for Nn, mean_mc, sd_mc, tt in mc_rows:
    print(f"{'Monte Carlo, N = ' + f'{Nn:,}':>24}  {mean_mc:12.4f}  {sd_mc:16.4f}  {tt:8.2f}")
print(f"{'Young':>24}  {mean_assets(lam_y, gri):12.4f}  {0:16.4f}  {t_y:8.2f}")

**Check:** Young's method is exact to the grid and takes about as long as the smallest simulation, whose mean
is off by a quarter in either direction. A simulation that matches Young to two digits needs $N = 10^5$ and a
hundred times the time. And Young's $A(r)$ is a smooth function of $r$, which part 3 needs.

**The three pieces of $A(r)$.** The same $A(r)$ that part 3 calls in every bisection step: solve the household,
build $\mathbf{Q}_{\mathbf{g}}$, find $\boldsymbol{\lambda}$. Which one is expensive?

In [ ]:
(C_, G_, it_), t_hh = best_of(lambda: solve_household(r0, w0, par, mpar, gri))
Q_, t_Q = best_of(lambda: transition_matrix(G_, gri))
(lam_, it_lam), t_lam = best_of(lambda: stationary_dist(Q_, mpar, gri))
print(f"solve the household:          {1000 * t_hh:6.0f} ms  ({it_} EGM iterations, {1000 * t_hh / it_:.2f} ms each)")
print(f"build Q:                      {1000 * t_Q:6.0f} ms")
print(f"stationary distribution:      {1000 * t_lam:6.0f} ms  ({it_lam} iterations, {1000 * t_lam / it_lam:.3f} ms each)")
print(f"total A(r):                   {1000 * (t_hh + t_Q + t_lam):6.0f} ms")

**Check:** the household and the distribution each take a few hundred iterations of a cheap step, and
together they are the bill. Building $\mathbf{Q}_{\mathbf{g}}$ is a few milliseconds. (Julia: run the cell twice, the
first run includes compilation. `@time solve_household(r0, w0, par, mpar, gri)` also prints the allocations,
and a step that allocates a fresh array every iteration is the first thing to look at.)

**The profiler.** Python's `cProfile` records every function call and `pstats` sorts the result. Two columns
matter: `tottime` is the time spent inside the function itself, `cumtime` includes everything it calls. Sort by
`tottime` to find the hot spot, by `cumtime` to see the structure. Julia's `@profile` samples the call stack
many times per second and `Profile.print()` prints the tree, with the sample counts per line.

In [ ]:
pr = cProfile.Profile()
pr.enable()
C_, G_, _ = solve_household(r0, w0, par, mpar, gri)
lam_, _ = stationary_dist(transition_matrix(G_, gri), mpar, gri)
pr.disable()
stats = pstats.Stats(pr)
stats.sort_stats("tottime").print_stats(8)      # the eight functions with the most time inside them

**Reading the output.** In Python the top entries are `egm_step` itself (the elementwise work, $369$ calls),
the sparse matrix-vector product (called $1{,}267$ times), and `np.interp` inside `egm_step` ($7 \times 369$
calls). Nothing surprising, and nothing that a different line of code fixes: the heavy lifting already happens in
library calls. What is left is **fewer iterations**, and one way to fewer iterations is a better starting point.

**What that means for the code.** The profile confirms what the code already is: the heavy lifting happens
in library calls on whole arrays, the matrix is sparse, the solver is EGM. Nothing inside the code buys much.
Fewer iterations would, and part 3 uses the one cheap way to get them: inside the equilibrium loop, $A(r)$ is
evaluated at rates that get closer and closer, and the policy and the distribution at the previous $r$ are good
starting points for the next (`C0` and `lam0`). That saves about a quarter of the iterations, no more, because
both loops converge linearly: the count is $\ln(\text{start} / \text{tolerance}) / \ln(\text{rate})$, and a
start a hundred times closer removes two of eight orders of magnitude. The big factors were decided earlier, by
the algorithm: EGM against VFI ($0.04$ s against $3$), Young against Monte Carlo ($0.03$ against $7$).

### 3. One equilibrium wrapper for Huggett and Aiyagari 

*Session 6, slides 18 to 22.*

Both equilibria are one equation in one unknown, the interest rate: households supply assets, something
demands them, and the rate adjusts until supply equals demand. What differs is **who demands, and what the
wage is**:

| | Huggett (1993) | Aiyagari (1994) |
| --- | --- | --- |
| Asset | a bond, lent and borrowed between households | capital, rented to a firm |
| Borrowing limit $\underline{a}$ | $-2$ | $0$ |
| Wage $w(r)$ | $1$ | $(1 - \alpha) (K^d(r) / L)^\alpha$, from the firm |
| Demand $D(r)$ | $0$ (zero net supply) | $K^d(r) = L \bigl( (r + \delta) / \alpha \bigr)^{1 / (\alpha - 1)}$ |
| Equilibrium | $A(r^*) = 0$ | $A(r^*) = K^d(r^*)$ |

So one wrapper does both: an **economy** is a borrowing limit, a wage function, and a demand function. The
**distribution block** `aggregate` turns $(r, w)$ into $A(r)$ (household, $\mathbf{Q}_{\mathbf{g}}$,
$\boldsymbol{\lambda}$, sum), and `stationary_equilibrium` bisects on the excess supply $A(r) - D(r)$, which
increases in $r$, passing the warm start of part 2 from one step to the next. Keep the bracket below
$1 / \beta - 1$, where a patient household accumulates forever and $\boldsymbol{\lambda}$ does not exist.

In [ ]:
@dataclass
class Aggregates:
    A: float                  # mean assets A(r), the households' asset supply
    lam: np.ndarray           # the stationary distribution, N_a x N_z
    C: np.ndarray             # consumption policy
    G: np.ndarray             # savings policy


def aggregate(r, w, gri, warm=None):
    # The distribution block: household -> policy chain -> stationary distribution -> mean assets, at the prices (r, w).
    # warm: the Aggregates of a nearby r, used as the starting point (part 2), or None for a cold start
    C, G, _ = solve_household(r, w, par, mpar, gri, C0=None if warm is None else warm.C)
    Q = transition_matrix(G, gri)
    lam, _ = stationary_dist(Q, mpar, gri, lam0=None if warm is None else warm.lam)
    return Aggregates(A=mean_assets(lam, gri), lam=lam, C=C, G=G)

In [ ]:
@dataclass
class Economy:
    name: str
    gri: Grids                # the grids, with the borrowing limit of this economy
    wage: Callable            # w(r)
    demand: Callable          # D(r): what absorbs the households' assets at the rate r


def stationary_equilibrium(eco, lo, hi, tol):
    # Bisection on the excess supply A(r) - D(r), which increases in r, with warm starts along the way. Returns r*, the
    # aggregates at r*, and the step count
    warm, k = None, 0
    while hi - lo > tol:
        mid = (lo + hi) / 2
        warm = aggregate(mid, eco.wage(mid), eco.gri, warm)   # A(mid), started from the previous step's solution
        excess = warm.A - eco.demand(mid)
        # excess supply increases in r: if it is negative the root is above mid, keep [mid, hi], otherwise [lo, mid]
        lo, hi = (mid, hi) if excess < 0 else (lo, mid)
        k += 1
    return (lo + hi) / 2, warm, k

**Huggett (slides 18 and 19).** $w = 1$, nothing demands the bond, $\underline{a} = -2$. $A(r)$ increases in $r$
and explodes as $r \to 1 / \beta - 1$, so the bracket $[-0.02, 0.04]$ is safe. The inner tolerances ($10^{-8}$ on
the policy, $10^{-12}$ on the distribution) sit below the outer one ($10^{-6}$ on $r$), since their stopping
errors enter $A(r)$.

In [ ]:
huggett = Economy(name="Huggett", gri=make_grids(-2.0, par, mpar), wage=lambda r: 1.0, demand=lambda r: 0.0)
for r in (-0.02, 0.0, 0.02, 0.04):
    print(f"r = {100 * r:5.1f} percent: A(r) = {aggregate(r, 1.0, huggett.gri).A:8.3f}")

In [ ]:
t0 = time.perf_counter()
rH, aggH, kH = stationary_equilibrium(huggett, -0.02, 0.04, 1e-6)
print(f"Huggett: r* = {100 * rH:.3f} percent after {kH} bisection steps, {time.perf_counter() - t0:.1f} s, A(r*) = {aggH.A:.1e}")
print(f"share at the limit a = -2: {aggH.lam[0].sum():.3f}, and 1/beta - 1 = {100 * (1 / par.beta - 1):.2f} percent for comparison")

**Check (slides 18 and 19):** $A(-0.02) \approx -1.1$ and $A(0.04) \approx 46$ bracket the root; $16$ steps
($\log_2(0.06 / 10^{-6}) \approx 16$) to $r^* = 0.25$ percent with $|A(r^*)| < 10^{-4}$, in well under a
second. $r^*$ is far below $1 / \beta - 1 = 4.17$ percent: households save for precaution, and someone has to
hold the savings, so the rate falls until $14$ percent of them sit at the borrowing limit.

**Aiyagari (slides 20 to 22).** Households with $\underline{a} = 0$ and income $w z$, where $z$
is labour efficiency, so aggregate labour is $L = E[z]$. A firm with $Y = K^\alpha L^{1 - \alpha}$,
$\alpha = 0.36$, $\delta = 0.08$, rents capital and labour at competitive prices:

$$
r = \alpha (K / L)^{\alpha - 1} - \delta, \qquad w = (1 - \alpha) (K / L)^\alpha .
$$

The first equation, solved for $K$, is **capital demand** $K^d(r)$, decreasing in $r$. The second gives the wage
that goes with it. The only new step inside the loop: compute $w(r)$ from $K^d(r)$ before solving the household
problem, which the `wage` field of the economy does.

In [ ]:
alpha, delta = 0.36, 0.08
L = gri.lam_z @ gri.z                           # aggregate labour: mean efficiency E[z]


def K_demand(r):
    # capital demand of the firm: r = alpha (K/L)^(alpha - 1) - delta, solved for K
    # K = L ((r + delta) / alpha)^(1 / (alpha - 1)): the formula on session 6, slide 20
    return L * ((r + delta) / alpha) ** (1 / (alpha - 1))


def wage(r):
    # the wage that goes with capital demand at r: w = (1 - alpha) (K/L)^alpha
    return (1 - alpha) * (K_demand(r) / L) ** alpha

In [ ]:
aiyagari = Economy(name="Aiyagari", gri=make_grids(0.0, par, mpar), wage=wage, demand=K_demand)
r_cm = 1 / par.beta - 1                         # the complete markets rate: no precautionary saving, r = 1/beta - 1
K_cm = K_demand(r_cm)
print(f"complete markets benchmark: r = {100 * r_cm:.2f} percent, K = {K_cm:.2f}, K/Y = {K_cm / (K_cm ** alpha * L ** (1 - alpha)):.2f}, w = {wage(r_cm):.3f}")

In [ ]:
t0 = time.perf_counter()
rA, aggA, kA = stationary_equilibrium(aiyagari, 0.0, r_cm - 1e-4, 1e-6)
KA = aggA.A
YA = KA ** alpha * L ** (1 - alpha)
print(f"Aiyagari: r* = {100 * rA:.2f} percent after {kA} steps, {time.perf_counter() - t0:.1f} s")
print(f"K* = {KA:.2f} (demand {K_demand(rA):.2f}), w* = {wage(rA):.3f}, K/Y = {KA / YA:.2f}, share at the limit {aggA.lam[0].sum():.3f}")
print(f"against complete markets: K = {K_cm:.2f}, K/Y = {K_cm / (K_cm ** alpha * L ** (1 - alpha)):.2f}: precautionary saving raises capital by {100 * (KA / K_cm - 1):.0f} percent")

**Check (slide 21):** $r^* = 2.53$ percent, $K^* = 7.6$, $K / Y = 3.4$, $2.6$ percent of households at
$\underline{a} = 0$. Under complete markets ($r = 1 / \beta - 1 = 4.17$ percent) $K = 6.0$ and $K / Y = 3.0$:
precautionary saving raises the capital stock by a quarter, the point of the paper.

**What the warm starts bought.** Run the Aiyagari bisection once more with cold starts: the same
wrapper, but `aggregate` started from scratch in every step. Compare the time, and the answer.

### 4. Experiment: the settings of the solver, and what they cost

*Session 8, slides 2 and 3.*

Everything above ran with one set of numerical settings: $N_a = 200$ grid points, the policy iterated to
$10^{-8}$, the distribution to $10^{-12}$, the bisection on $r$ to $10^{-6}$. None of these numbers is sacred.
The cell below runs the three pieces of $A(r)$ and the whole Aiyagari equilibrium under five settings, one
change at a time, and prints one line per setting: the time of one $A(r)$, the two iteration counts, the
answer $r^*$ and $K^*$, and the time of the equilibrium.

Read the table with three questions in mind:

1. **The grid.** Doubling $N_a$ doubles the work of every EGM step and every $\mathbf{Q}_{\mathbf{g}}^\top
   \boldsymbol{\lambda}$ product. Does the time double? Do the iteration counts move? Does $r^*$ move, and in
   which digit? (That digit is the grid error of the baseline.)
2. **The tolerances.** Both inner loops converge linearly, so the count grows with the logarithm of the
   tolerance: $10^{-8}$ to $10^{-10}$ should add about as many iterations as $10^{-6}$ to $10^{-8}$. Check.
   Then compare the answers: does $r^*$ move when the policy tolerance is $10^{-6}$, and by how much relative
   to the outer tolerance of $10^{-6}$ on $r$? This is the tolerance hierarchy of the slides: the inner
   stopping errors enter $A(r)$, so they must sit below the outer one.
3. **Where the time goes.** Which setting buys the most time, and what does it cost in digits? The slides say:
   the algorithm sets the order of magnitude, the code the factor. Where in this table is the factor, and
   where would you need another algorithm?

Then change the list yourself. Candidates: $N_z = 5$ or $11$ (the income chain; note that $L = E[z]$ then
changes too, so recompute it), `amax` $= 50$ or $200$ at the same $N_a$ (the spacing near the limit changes),
the bisection tolerance $10^{-4}$ instead of $10^{-6}$, or only the distribution tolerance loosened to
$10^{-8}$ with the policy tolerance kept. Write the setting next to every number you report, as the slides say.


In [ ]:
from dataclasses import replace

mpar0 = mpar                                    # the baseline settings, restored at the end
settings = [                                    # name, N_a, policy tolerance, distribution tolerance
    ("baseline", 200, 1e-8, 1e-12),
    ("coarser grid, N_a = 100", 100, 1e-8, 1e-12),
    ("finer grid, N_a = 400", 400, 1e-8, 1e-12),
    ("looser tolerances", 200, 1e-6, 1e-8),
    ("tighter tolerances", 200, 1e-10, 1e-14),
]
print(f"{'setting':<24} {'A(r) ms':>8} {'EGM it':>7} {'dist it':>8} {'r* (%)':>8} {'K*':>7} {'eq. s':>6}")
for name, na, crit_pol, crit_dist in settings:
    mpar = replace(mpar0, na=na, crit_pol=crit_pol, crit_dist=crit_dist)   # aggregate reads the global mpar
    g = make_grids(0.0, par, mpar)              # the grid of this setting, borrowing limit 0 as in Aiyagari
    (C_, G_, it_), t_hh = best_of(lambda: solve_household(r0, w0, par, mpar, g))
    (lam_, it_lam), t_lam = best_of(lambda: stationary_dist(transition_matrix(G_, g), mpar, g))
    eco = Economy(name="Aiyagari", gri=g, wage=wage, demand=K_demand)
    t0 = time.perf_counter()
    r_, agg_, _ = stationary_equilibrium(eco, 0.0, r_cm - 1e-4, 1e-6)
    t_eq = time.perf_counter() - t0
    print(f"{name:<24} {1000 * (t_hh + t_lam):8.0f} {it_:7d} {it_lam:8d} {100 * r_:8.3f} {agg_.A:7.3f} {t_eq:6.2f}")
mpar = mpar0                                    # back to the baseline for anything you run below


**What to expect, roughly.** The time of $A(r)$ grows with $N_a$, less than proportionally at these sizes
(the fixed cost of every library call does not grow), while the iteration counts hardly move; $r^*$ changes in the third digit between $N_a = 100$ and $400$, which says how much of the
baseline answer is grid. The tolerances move the counts by a roughly constant number of iterations per factor
$100$, and the answer by less than the outer tolerance when the inner ones sit below it, by more when they do
not. The equilibrium time is the $A(r)$ time times the bisection steps, minus what the warm starts save. If your
numbers say something else, that is the interesting case: find out why.


**At home, if you like.** (i) The wealth distribution at the Aiyagari equilibrium (session 6, slide 22):
the Lorenz curve and the Gini coefficient from `aggA.lam`, mean $7.6$, median $5.3$, the top $10$ percent own
$32$ percent, Gini $0.52$. (ii) Run the Aiyagari bisection once more with `aggregate` started cold in every
step (no `warm` argument) and compare the time: the warm starts save about a quarter of the iterations, no
more, because both inner loops converge linearly. (iii) A third economy for the wrapper: Huggett with a
government that supplies bonds $B = 0.5$, so `demand = lambda r: 0.5`. Where does $r^*$ move, and why?